## Imports

In [1]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 100)

## Importing scored outputs

In [3]:
df = pd.read_pickle('../data/pd_model_scored_output.pkl')
df.shape

(40000, 121)

In [6]:
df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'total_rec_late_fee',
 'last_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',

In [8]:
df[['actual_default_flag','pd_score','pd_prediction']].tail(10)

,actual_default_flag,pd_score,pd_prediction
692153,0,0.468976,0
1816200,0,0.233568,0
1270714,0,0.050490,0
1709748,0,0.533128,1
596968,0,0.739552,1
1135476,1,0.222382,0
1601713,1,0.950497,1
344609,0,0.146255,0
874428,1,0.554675,1
1910437,1,0.368561,0


In [9]:
df['pd_score'].describe()

count    40000.000000
mean         0.438078
std          0.215661
min          0.025880
25%          0.263283
50%          0.421281
75%          0.598333
max          0.967616
Name: pd_score, dtype: float64

## Risk Segmentation - Quantile Segementation

In [12]:
df['risk_band'] = pd.qcut(df['pd_score'], q=5, labels=['Very Low','Low', 'Medium', 'High', 'Very High'])
df[['pd_score','risk_band']].head(10)

,pd_score,risk_band
824713,0.495415,High
591446,0.504460,High
1789024,0.301881,Low
1084390,0.685799,Very High
1807296,0.739917,Very High
1286568,0.721815,Very High
713788,0.553942,High
2050348,0.275931,Low
1266891,0.943725,Very High
1271617,0.363590,Medium


In [13]:
df['risk_band'].value_counts().sort_index()

risk_band
Very Low     8000
Low          8000
Medium       8000
High         8000
Very High    8000
Name: count, dtype: int64

In [19]:
risk_band_summary = df.groupby('risk_band').agg(
    borrower_count=('actual_default_flag', 'count'),
    default_count=('actual_default_flag', 'sum'),
    default_rate=('actual_default_flag', 'mean'),
    min_pd_score=('pd_score','min'),
    max_pd_score=('pd_score','max'),
    avg_pd_score=('pd_score','mean')
).reset_index()

risk_band_summary

/var/folders/99/nqyz4s796bx09s6p71mvxd5h0000gn/T/ipykernel_19372/938080201.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_band_summary = df.groupby('risk_band').agg(


,risk_band,borrower_count,default_count,default_rate,min_pd_score,max_pd_score,avg_pd_score
0,Very Low,8000,389,0.048625,0.025880,0.230880,0.155090
1,Low,8000,860,0.107500,0.230932,0.358714,0.294819
2,Medium,8000,1473,0.184125,0.358716,0.486674,0.422031
3,High,8000,2111,0.263875,0.486684,0.641309,0.560484
4,Very High,8000,4027,0.503375,0.641349,0.967616,0.757968


## Create Scorecard style Risk
#### High Score -> Lower Risk

In [26]:
df['risk_score'] = ((1-df['pd_score'])*1000).round(0)
df[['pd_score','risk_score','risk_band']].head(10)

,pd_score,risk_score,risk_band
824713,0.495415,505.0,High
591446,0.504460,496.0,High
1789024,0.301881,698.0,Low
1084390,0.685799,314.0,Very High
1807296,0.739917,260.0,Very High
1286568,0.721815,278.0,Very High
713788,0.553942,446.0,High
2050348,0.275931,724.0,Low
1266891,0.943725,56.0,Very High
1271617,0.363590,636.0,Medium


## Risk Score Summary By Risk Band

In [31]:
score_summary = df.groupby('risk_score').agg(
    borrower_count=('risk_score','count'),
    avg_risk_score=('risk_score','mean'),
    min_risk_score=('risk_score','min'),
    max_risk_score=('risk_score','max'),
    actual_default_rate=('actual_default_flag','mean')
).reset_index()

score_summary

,risk_score,borrower_count,avg_risk_score,min_risk_score,max_risk_score,actual_default_rate
0,32.0,1,32.0,32.0,32.0,1.00
1,36.0,1,36.0,36.0,36.0,1.00
2,38.0,1,38.0,38.0,38.0,1.00
3,39.0,2,39.0,39.0,39.0,1.00
4,40.0,4,40.0,40.0,40.0,0.75
...,...,...,...,...,...,...
931,968.0,1,968.0,968.0,968.0,0.00
932,969.0,6,969.0,969.0,969.0,0.00
933,970.0,1,970.0,970.0,970.0,0.00
934,971.0,3,971.0,971.0,971.0,0.00


## Loan Decision Preview Based on PD Risk Band

In [32]:
df['risk_band'].unique()

['High', 'Low', 'Very High', 'Medium', 'Very Low']
Categories (5, object): ['Very Low' < 'Low' < 'Medium' < 'High' < 'Very High']

In [33]:
def preliminary_decision(risk_band):
    if risk_band in ['Very Low','Low']:
        return 'Approve'
    elif risk_band == 'Medium':
        return 'Manual Review'
    else:
        return 'Reject'

In [34]:
df['preliminary_decision']=df['risk_band'].apply(preliminary_decision)
df['preliminary_decision'].value_counts()

preliminary_decision
Reject           16000
Approve          16000
Manual Review     8000
Name: count, dtype: int64

## Decision Summary

In [35]:
decision_summary = df.groupby('preliminary_decision').agg(
    borrower_count=('actual_default_flag', 'count'),
    avg_pd_score=('pd_score','mean'),
    avg_risk_score=('risk_score', 'mean'),
    actual_default_rate=('actual_default_flag','mean')
).reset_index()

decision_summary

,preliminary_decision,borrower_count,avg_pd_score,avg_risk_score,actual_default_rate
0,Approve,16000,0.224955,775.046812,0.078063
1,Manual Review,8000,0.422031,577.970250,0.184125
2,Reject,16000,0.659226,340.774688,0.383625


## Save Outputs

In [37]:
df.to_pickle('../data/risk_segmented_output.pkl')
risk_band_summary.to_csv('../data/risk_band_summary.csv', index=False)
score_summary.to_csv('../data/score_summary.csv', index=False)
decision_summary.to_csv('../data/decision_summary.csv', index=False)